# RGV Precipitation Prediction — Master Pipeline
**IMERG 2000–2021 | XGBoost | Rio Grande Valley**

Run cells top to bottom. Each section calls the src scripts and displays results inline.

## 0. Setup & Imports

In [ ]:
import subprocess, sys, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import xgboost as xgb
from pathlib import Path
from sklearn.metrics import mean_absolute_error, r2_score

# Helper to run a script and stream output live
def run_script(path):
    print(f"\n{'='*50}\nRunning: {path}\n{'='*50}")
    result = subprocess.run([sys.executable, path], capture_output=False, text=True)
    if result.returncode != 0:
        print(f"ERROR in {path}")

print("Setup complete.")

## 1. Data Loader — .nc → .parquet (RGV crop)

In [ ]:
run_script("src/data_loader.py")

## 2. Preprocess — Clean, lag features, seasonal encoding

In [ ]:
run_script("src/preprocess.py")

### Quick sanity check on final training data

In [ ]:
files = glob.glob("./data/final_training_set/*.parquet")
df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)

print(f"Total rows:  {len(df):,}")
print(f"Columns:     {df.columns.tolist()}")
print(f"Year range:  {int(df['year'].min())} – {int(df['year'].max())}")
print(f"\nMissing values:\n{df.isnull().sum()}")
df.describe()

## 3. Train Model

In [ ]:
run_script("src/model.py")

## 4. Evaluate Model

In [ ]:
run_script("src/evaluate.py")

### View output plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, img_path, title in zip(
    axes,
    ["feature_importance.png", "actual_vs_predicted.png"],
    ["Feature Importance", "Actual vs Predicted"]
):
    if Path(img_path).exists():
        ax.imshow(mpimg.imread(img_path))
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.axis('off')
    else:
        ax.text(0.5, 0.5, f'{img_path} not found', ha='center')

plt.tight_layout()
plt.show()

## 5. Future Prediction — 2022

In [ ]:
MODEL_PATH = Path("./models/rgv_precipitation_model.json")

# Build the features used in training
# Update this list to match whatever features your model.py uses
FEATURES = ['lat', 'lon', 'month', 'day_of_year']

# Generate future grid for 2022
lat_vals = np.arange(25.0, 27.5, 0.1)
lon_vals = np.arange(-100.5, -96.5, 0.1)
days = pd.date_range("2022-01-01", "2022-12-31", freq="D")

rows = []
for day in days:
    for lat in lat_vals:
        for lon in lon_vals:
            rows.append({
                'lat': round(lat, 2),
                'lon': round(lon, 2),
                'month': day.month,
                'day_of_year': day.day_of_year,
                'date': day
            })

future_df = pd.DataFrame(rows)

model = xgb.XGBRegressor()
model.load_model(MODEL_PATH)

future_df['predicted_precipitation_mm_hr'] = model.predict(future_df[FEATURES])
future_df['predicted_precipitation_mm_hr'] = future_df['predicted_precipitation_mm_hr'].clip(lower=0)

output_path = "./data/future_predictions_2022.parquet"
future_df.to_parquet(output_path, index=False)
print(f"Saved {len(future_df):,} predictions to {output_path}")
future_df.head()

### Plot monthly average predicted precipitation for 2022

In [ ]:
monthly = future_df.groupby('month')['predicted_precipitation_mm_hr'].mean()

month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(month_names, monthly.values, color='steelblue', edgecolor='white', linewidth=0.5)
ax.set_title("Predicted Monthly Avg Precipitation — RGV 2022", fontsize=14, fontweight='bold')
ax.set_ylabel("Precipitation (mm/hr)")
ax.set_xlabel("Month")

for bar, val in zip(bars, monthly.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig("predicted_2022_monthly.png", dpi=150)
plt.show()
print("Saved predicted_2022_monthly.png")